# 02 — Ingesta a Bronze
**Spec:** docs/specs/02_bronze_ingesta.md · **Requisitos:** R-06, R-07, R-08 · **Escribe:** `iol_bronze.trades_raw`, `iol_ops.ingestion_runs`

Guarda las columnas originales como texto, con linaje. El MERGE usa `(id_transaccion, record_hash)` y nunca actualiza (ADR-002).
`demo_rerun = true` corre la ingesta dos veces para demostrar la idempotencia. El orquestador la corre una vez.

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("demo_rerun", "true", ["true", "false"])

import json
import os
import sys
import uuid

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import bronze, config, io

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
RUNS = config.BRONZE_DEMO_RUNS if dbutils.widgets.get("demo_rerun") == "true" else 1
print("pipeline_run_id:", RUN_ID)

io.ensure_schemas(spark, catalog=CATALOG)
trades_table = io.table_name("bronze", "trades_raw", catalog=CATALOG)
runs_table = io.table_name("ops", "ingestion_runs", catalog=CATALOG)
csv_files = bronze.list_csv_files(io.landing_path("trades", catalog=CATALOG))
print("Archivos:", csv_files)

pipeline_run_id: 29de8374-d675-4813-a72e-86fd054965c7
Archivos: ['/Volumes/workspace/iol_bronze/landing/trades/data-set-challenge-5-.csv']


## Ingesta

In [0]:
results = [bronze.ingest_trades(spark, path, trades_table, runs_table, RUN_ID) for _ in range(RUNS) for path in csv_files]
for r in results:
    print({k: r[k] for k in ("status", "rows_read", "rows_inserted", "rows_already_present", "rows_conflict", "rows_duplicate_in_file", "duplicate_row_numbers", "error")})

{'status': 'SUCCESS', 'rows_read': 100000, 'rows_inserted': 100000, 'rows_already_present': 0, 'rows_conflict': 0, 'rows_duplicate_in_file': 0, 'duplicate_row_numbers': [], 'error': None}
{'status': 'SUCCESS', 'rows_read': 100000, 'rows_inserted': 0, 'rows_already_present': 100000, 'rows_conflict': 0, 'rows_duplicate_in_file': 0, 'duplicate_row_numbers': [], 'error': None}


## Corridas registradas
`rows_read = rows_inserted + rows_already_present + rows_duplicate_in_file`; `rows_conflict` es parte de `rows_inserted`.
Esta tabla crece una fila por corrida, a propósito. La que queda idéntica es `trades_raw`.

In [0]:
if spark.catalog.tableExists(runs_table):
    display(spark.table(runs_table).orderBy("started_at"))
else:
    print(f"{runs_table} no existe: no hubo archivos para ingerir.")

ingestion_run_id,pipeline_run_id,source_file,source_file_hash,rows_read,rows_inserted,rows_already_present,rows_conflict,rows_duplicate_in_file,duplicate_row_numbers,status,error,started_at,ended_at
8caaad05-8465-410e-94d5-c956f33eb914,29de8374-d675-4813-a72e-86fd054965c7,/Volumes/workspace/iol_bronze/landing/trades/data-set-challenge-5-.csv,dd9e9b9694c33810f02a85e6418bc7b373f8c84430a76d24141b8be9e10f36c8,100000,100000,0,0,0,List(),SUCCESS,null,2026-11-01T15:24:47.478Z,2026-11-01T15:25:21.425Z
475b9053-fef0-4597-977b-3694f6cbcc22,29de8374-d675-4813-a72e-86fd054965c7,/Volumes/workspace/iol_bronze/landing/trades/data-set-challenge-5-.csv,dd9e9b9694c33810f02a85e6418bc7b373f8c84430a76d24141b8be9e10f36c8,100000,0,100000,0,0,List(),SUCCESS,null,2026-11-01T15:25:28.718Z,2026-11-01T15:25:42.794Z


## Tabla Bronze

In [0]:
rows_out = 0
if spark.catalog.tableExists(trades_table):
    rows_out = spark.table(trades_table).count()
    print(f"{trades_table}: {rows_out:,} filas")
    detail = spark.sql(f"DESCRIBE DETAIL {trades_table}").first()
    print("format:", detail["format"], "· numFiles:", detail["numFiles"], "· sizeInBytes:", detail["sizeInBytes"])
    display(spark.table(trades_table).select("source_file", "source_file_hash", "schema_version", "ingestion_run_id").distinct())
else:
    print(f"{trades_table} no existe: ninguna corrida escribió filas. El motivo está en la tabla de corridas.")
orphans = bronze.orphan_ingestion_runs(spark, trades_table, runs_table)
print("Corridas de Bronze sin registro en ingestion_runs:", orphans or "ninguna")

workspace.iol_bronze.trades_raw: 100,000 filas
format: delta · numFiles: 1 · sizeInBytes: 6192883


source_file,source_file_hash,schema_version,ingestion_run_id
/Volumes/workspace/iol_bronze/landing/trades/data-set-challenge-5-.csv,dd9e9b9694c33810f02a85e6418bc7b373f8c84430a76d24141b8be9e10f36c8,trades_v1,8caaad05-8465-410e-94d5-c956f33eb914


Corridas de Bronze sin registro en ingestion_runs: ninguna


In [0]:
ok = bool(results) and all(r["status"] == "SUCCESS" for r in results) and not orphans
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if ok else "FAILED", "pipeline_run_id": RUN_ID, "rows_out": rows_out,
                                  "runs": [{"status": r["status"], "rows_inserted": r["rows_inserted"]} for r in results]}))